In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import wget
import tqdm
import os
import zipfile
import torch
import torch.optim as optim
import random

from data import ShapeNetClassficationDataset
from PointNet import PointNetCls, PointNetSeg, PointNetClsTrainer, PointNetSegTrainer

In [ ]:
# shapenetcore_partanno_segmentation

url = 'https://huggingface.co/datasets/wangps/shapenet_segmentation/resolve/main/shapenetcore_partanno_segmentation_benchmark_v0_normal.zip'
data_root = '../../Data/shapenetcore_partanno_segmentation'

wget.download(url, data_root)
zip_path=os.path.join(data_root,"shapenetcore_partanno_segmentation_benchmark_v0_normal.zip")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_root)
os.remove(zip_path)

In [ ]:
random.seed(0)
torch.manual_seed(0)

BATCH_SIZE=32
NUM_WORKERS=1

model_save_root = "../../Models/PointNet"
os.makedirs(model_save_root,exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

In [ ]:
train_dataset = ShapeNetClassficationDataset(
        root=os.path.join(data_root,"shapenetcore_partanno_segmentation_benchmark_v0_normal"),
        split='train',
        npoints=1024,
        with_data_augmentation=True)

test_dataset = ShapeNetClassficationDataset(
        root=os.path.join(data_root,"shapenetcore_partanno_segmentation_benchmark_v0_normal"),
        split='test',
        npoints=1024,
        with_data_augmentation=False)

train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=int(NUM_WORKERS))

test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=int(NUM_WORKERS))

print(len(train_dataset), len(test_dataset))
num_classes = len(train_dataset.classes)
print('classes', num_classes)

# PointNet

In [ ]:
classifier = PointNetCls(k=num_classes)
optimizer = optim.Adam(classifier.parameters(), lr=0.01, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.8)



In [ ]:
torch.save(classifier.state_dict(), pjoin(opt.expf, f'cls_{args.dim}D/model.pth'))

#### Evaluation

In [ ]:
total_correct = 0
total_testset = 0
for i,data in tqdm.tqdm(enumerate(test_dataloader, 0)):
    points, target = data
    target = target[:, 0]
    classifier = classifier.eval()
    pred, _ = classifier(points)
    pred_choice = pred.data.max(1)[1]
    correct = pred_choice.eq(target.data).cpu().sum()
    total_correct += correct.item()
    total_testset += points.size()[0]

print("final accuracy {}".format(total_correct / float(total_testset)))